# Previsão de consumo (regressão linear)

Prevê, para um usuário, o **consumo (kWh)** e o **valor da fatura (R$)** do mês seguinte,
com base no histórico do mês atual e atributos do plano/ponto de recarga habitual.

## Abordagem de dataset

Janela deslizante: cada linha é uma transição `(usuário, mês N -> mês N+1)`. Com 6 meses de
dado (mar-ago/2026), cada usuário contribui com até 5 exemplos (mar→abr, abr→mai, ..., jul→ago).

**Cuidado metodológico**: linhas do mesmo usuário são correlacionadas entre si (não são
observações independentes). Para não inflar artificialmente a métrica de avaliação, o split
treino/teste é feito **por usuário** um usuário nunca aparece nos dois conjuntos ao mesmo tempo.

## Modelos

Dois modelos treinados separadamente (mais simples de interpretar e debugar do que uma saída
dupla forçada):
- `consumo_kwh_previsao.joblib` — prevê `kwh_proximo_mes`
- `fatura_valor_previsao.joblib` — prevê `valor_fatura_proximo_mes`

## Features

- **Histórico do usuário no mês atual**: `kwh_mes_atual`, `duracao_media_min`, `n_sessoes_mes`,
  `tendencia_kwh` (kwh_mes_atual - kwh_mes_anterior; 0 no primeiro mês de cada usuário)
- **Plano**: `plan_type` (one-hot)
- **Ponto de recarga habitual**: `power_kw`


## Setup

In [1]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import psycopg2
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)

In [2]:
DB_CONFIG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": os.getenv("POSTGRES_PORT", "5432"),
    "dbname": os.getenv("POSTGRES_DB", "evchargeops"),
    "user": os.getenv("POSTGRES_USER", "evchargeops"),
    "password": os.getenv("POSTGRES_PASSWORD", ""),
}

MODELS_DIR = Path.cwd().parent / "models" if (Path.cwd().name == "models") else Path.cwd() / "models"
MODELS_DIR = Path("models") if Path("models").exists() else Path(".")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_FEATURES = ["kwh_mes_atual", "duracao_media_min", "n_sessoes_mes", "tendencia_kwh", "power_kw"]
CATEGORICAL_FEATURES = ["plan_type"]

## Carga dos dados

Agrega sessões por usuário/mês e traz atributos de plano/ponto.
    Uma linha por (usuário, mês) com uso — usuários sem sessão num mês
    não geram linha (não há 'mês seguinte' a prever para um mês vazio).

In [ ]:
def fetch_monthly_user_data(conn) -> pd.DataFrame:
    query = """
        SELECT
            s.user_id,
            d.ref_month,
            u.plan_type,
            p.power_kw,
            SUM(s.kwh_delivered) AS kwh_mes,
            AVG(s.duration_min) AS duracao_media_min,
            COUNT(*) AS n_sessoes_mes
        FROM fct_sessoes s
        JOIN dim_dates d ON d.session_date = s.session_date
        JOIN dim_users u ON u.user_id = s.user_id
        JOIN dim_points p ON p.point_id = u.point_id
        GROUP BY s.user_id, d.ref_month, u.plan_type, p.power_kw
        ORDER BY s.user_id, d.ref_month
    """
    return pd.read_sql(query, conn)


def fetch_invoice_by_user_month(conn) -> pd.DataFrame:
    query = "SELECT user_id, ref_month, total_amount FROM fct_invoices"
    return pd.read_sql(query, conn)

In [4]:
conn = psycopg2.connect(**DB_CONFIG)
try:
    monthly_df = fetch_monthly_user_data(conn)
    invoices_df = fetch_invoice_by_user_month(conn)
finally:
    conn.close()

print(f"{len(monthly_df)} combinações usuário/mês com uso.")
monthly_df.head()

840 combinações usuário/mês com uso.


/tmp/ipykernel_376119/620621503.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)
/tmp/ipykernel_376119/620621503.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,user_id,ref_month,plan_type,power_kw,kwh_mes,duracao_media_min,n_sessoes_mes
0,U0001,2026-03-01,pay_per_use,11.0,240.039,141.272727,11
1,U0001,2026-05-01,pay_per_use,11.0,98.149,107.166667,6
2,U0001,2026-06-01,pay_per_use,11.0,306.509,122.125000,16
3,U0001,2026-07-01,pay_per_use,11.0,121.353,100.000000,8
4,U0001,2026-08-01,pay_per_use,11.0,304.126,127.666667,15


## Construção do dataset de treino (janela deslizante)

Para cada usuário, cada par de meses consecutivos onde ele teve uso
    em ambos vira uma linha de treino (features do mês N, targets do mês N+1).

In [ ]:
def build_training_set(monthly_df: pd.DataFrame, invoices_df: pd.DataFrame) -> pd.DataFrame:
    monthly_df = monthly_df.sort_values(["user_id", "ref_month"]).copy()
    monthly_df["ref_month"] = pd.to_datetime(monthly_df["ref_month"])

    rows = []
    for user_id, group in monthly_df.groupby("user_id"):
        group = group.reset_index(drop=True)
        for i in range(len(group) - 1):
            current = group.iloc[i]
            next_row = group.iloc[i + 1]

            # Só forma par se os meses são consecutivos (não pula gaps —
            # um usuário sem uso no mês intermediário não gera uma
            # transição artificial "mês 1 -> mês 3").
            month_diff = (next_row["ref_month"].year - current["ref_month"].year) * 12 + \
                         (next_row["ref_month"].month - current["ref_month"].month)
            if month_diff != 1:
                continue

            tendencia = 0.0
            if i > 0:
                prev = group.iloc[i - 1]
                tendencia = current["kwh_mes"] - prev["kwh_mes"]

            fatura_next = invoices_df[
                (invoices_df["user_id"] == user_id) &
                (invoices_df["ref_month"] == next_row["ref_month"].date())
            ]["total_amount"]

            rows.append({
                "user_id": user_id,
                "kwh_mes_atual": current["kwh_mes"],
                "duracao_media_min": current["duracao_media_min"],
                "n_sessoes_mes": current["n_sessoes_mes"],
                "tendencia_kwh": tendencia,
                "power_kw": current["power_kw"],
                "plan_type": current["plan_type"],
                "kwh_proximo_mes": next_row["kwh_mes"],
                "valor_fatura_proximo_mes": fatura_next.iloc[0] if len(fatura_next) > 0 else None,
            })

    return pd.DataFrame(rows)

In [6]:
training_df = build_training_set(monthly_df, invoices_df)
print(f"{len(training_df)} linhas de treino ({training_df['user_id'].nunique()} usuários distintos).")
training_df.head()

653 linhas de treino (149 usuários distintos).


,user_id,kwh_mes_atual,duracao_media_min,n_sessoes_mes,tendencia_kwh,power_kw,plan_type,kwh_proximo_mes,valor_fatura_proximo_mes
0,U0001,98.149,107.166667,6,-141.890,11.0,pay_per_use,306.509,1660.90
1,U0001,306.509,122.125000,16,208.360,11.0,pay_per_use,121.353,680.00
2,U0001,121.353,100.000000,8,-185.156,11.0,pay_per_use,304.126,1627.75
3,U0002,387.580,111.636364,11,0.000,22.0,pacote_condominial,281.951,545.37
4,U0002,281.951,76.416667,12,-105.629,22.0,pacote_condominial,472.769,673.37


## Pipeline de treino e avaliação

In [7]:
def build_pipeline() -> Pipeline:
    preprocessor = ColumnTransformer([
        ("num", "passthrough", NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])
    return Pipeline([
        ("preprocess", preprocessor),
        ("regressor", LinearRegression()),
    ])


def train_and_evaluate(df: pd.DataFrame, target_col: str, label: str) -> Pipeline:
    data = df.dropna(subset=[target_col]).copy()
    X = data[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y = data[target_col]
    groups = data["user_id"]

    # Split por usuário (GroupShuffleSplit), não por linha aleatória —
    # garante que nenhum usuário apareça em treino E teste ao mesmo tempo.
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"--- {label} ---")
    print(f"Treino: {len(X_train)} linhas ({groups.iloc[train_idx].nunique()} usuários)")
    print(f"Teste:  {len(X_test)} linhas ({groups.iloc[test_idx].nunique()} usuários)")
    print(f"MAE  (erro médio absoluto): {mae:.2f}")
    print(f"R²   (variância explicada): {r2:.3f}")

    return pipeline

## Treino: consumo (kWh) do próximo mês

In [8]:
model_kwh = train_and_evaluate(training_df, "kwh_proximo_mes", "Modelo: consumo (kWh) do próximo mês")

--- Modelo: consumo (kWh) do próximo mês ---
Treino: 483 linhas (111 usuários)
Teste:  170 linhas (38 usuários)
MAE  (erro médio absoluto): 83.40
R²   (variância explicada): 0.470


## Treino: valor da fatura (R$) do próximo mês

In [9]:
model_fatura = train_and_evaluate(training_df, "valor_fatura_proximo_mes", "Modelo: valor da fatura (R$) do próximo mês")

--- Modelo: valor da fatura (R$) do próximo mês ---
Treino: 483 linhas (111 usuários)
Teste:  170 linhas (38 usuários)
MAE  (erro médio absoluto): 259.60
R²   (variância explicada): 0.324


## Persistência dos artefatos: Salvos em `models/`, para consumo posterior pela aba Streamlit (Etapa 8a).

In [10]:
joblib.dump(model_kwh, MODELS_DIR / "consumo_kwh_previsao.joblib")
joblib.dump(model_fatura, MODELS_DIR / "fatura_valor_previsao.joblib")
training_df.to_csv(MODELS_DIR / "consumo_training_data.csv", index=False)

print("Artefatos salvos em:", MODELS_DIR.resolve())
print(" -", "consumo_kwh_previsao.joblib")
print(" -", "fatura_valor_previsao.joblib")
print(" -", "consumo_training_data.csv")

Artefatos salvos em: /home/vinivaliati/projects/computer_science_fiap_challenge_2026/models
 - consumo_kwh_previsao.joblib
 - fatura_valor_previsao.joblib
 - consumo_training_data.csv
